# EDA Inicial - Inspección de Calidad
Este notebook realiza una exploración rápida del dataset original para identificar problemas de calidad y definir las reglas de transformación para dbt.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

## 1. Carga de Datos

In [ ]:
import os
# Ajustar el path ya que ahora estamos en la carpeta notebooks/
data_path = '../data/raw/ecommerce_customer_churn_dataset.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Dimensiones del dataset: {df.shape}")
    display(df.head())
else:
    print(f"Error: No se encontró el archivo en {os.path.abspath(data_path)}")

## 2. Estructura y Tipos

In [ ]:
df.info()

## 3. Calidad de Datos (Nulos y Duplicados)

In [ ]:
nulls = df.isnull().sum()
print("Valores nulos por columna:")
print(nulls[nulls > 0])

print(f"\nFilas duplicadas: {df.duplicated().sum()}")

## 4. Estadísticas Descriptivas

In [ ]:
df.describe().T

## 5. Distribución de la variable Target (Churned)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Churned', data=df, palette='viridis')
plt.title('Distribución de Clientes (Churned vs No Churn)')
plt.show()

## 6. Análisis de Correlación (Discovery de Variables Clave)

In [ ]:
# Heatmap de correlación contra el Target
plt.figure(figsize=(10, 8))
correlations = df.corr(numeric_only=True)['Churned'].sort_values(ascending=False)
sns.barplot(x=correlations.values, y=correlations.index, palette='RdYlBu')
plt.title('Influencia de las Variables en el Churn (Discovery)')
plt.show()

## 7. Interacción y Puntos de Dolor (Pain Points)

In [ ]:
# Comparación de Recency y Llamadas a Soporte
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(x='Churned', y='Days_Since_Last_Purchase', data=df, ax=axes[0])
axes[0].set_title('Recency vs Churn (Inactividad)')

sns.boxplot(x='Churned', y='Customer_Service_Calls', data=df, ax=axes[1])
axes[1].set_title('Soporte vs Churn (Fricción)')

plt.show()

## 8. Contexto de Calidad (Outliers e Inconsistencias)

In [ ]:
# Detección de anomalías en Edad y su relación con el LTV
plt.figure(figsize=(10, 4))
sns.scatterplot(x='Age', y='Lifetime_Value', hue='Churned', data=df)
plt.axvline(100, color='red', linestyle='--') 
plt.title('Identificación de Outliers en Edad (Contexto para dbt)')
plt.show()

print("Análisis de nulos por segmento:")
display(df.groupby('Churned').apply(lambda x: x.isnull().sum()).T)

## 9. Segmentación Estratégica (Geografía)

In [ ]:
# Tasa de Churn por País
country_churn = df.groupby('Country')['Churned'].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=country_churn.index, y=country_churn.values, palette='magma')
plt.ylabel('Tasa de Abandono (Promedio)')
plt.title('Riesgo de Abandono por País')
plt.show()

## 10. Engagement Digital

In [ ]:
# Impacto de la apertura de emails
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='Email_Open_Rate', hue='Churned', fill=True)
plt.title('Interacción con Emails (Engagement vs Fuga)')
plt.show()